In [ ]:
"""
load_ckd_xgboost_models.py
═══════════════════════════════════════════════════════════════════════════════
Standalone loader for the saved CKD XGBoost model package.

Usage
──────
    python load_ckd_xgboost_models.py --pkg_dir /home/ubuntu/xgboost_CKD_output/models
"""

import argparse, json, sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline


def load_package(pkg_dir: str | Path, map_to_cpu: bool = False) -> dict:
    """Load the saved XGBoost model package."""
    pkg = Path(pkg_dir)

    # Step 1. Register TwoTierImputer before joblib.load().
    src_dir = str(pkg / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    try:
        import two_tier_imputer as _tti
        import __main__
        __main__.TwoTierImputer = _tti.TwoTierImputer
        print(f"[LOAD] TwoTierImputer: {_tti.__file__}")
    except ImportError as e:
        raise ImportError(
            f"two_tier_imputer.py not found.\n"
            f"  Search path: {src_dir}\n"
            f"  Put two_tier_imputer.py in {src_dir}.\n"
            f"  Error: {e}"
        ) from e

    # Step 2. Metadata.
    with open(pkg / "metadata.json", encoding="utf-8") as f:
        metadata = json.load(f)
    model_order = metadata["model_order"]
    print(f"[LOAD] metadata: {len(model_order)} models | "
          f"horizon={metadata.get('horizon_days')} days | "
          f"model_type={metadata.get('model_type', 'xgboost')}")
    if "key_package_versions" in metadata:
        print(f"[LOAD] saved packages: "
              f"{list(metadata['key_package_versions'].values())}")

    # Step 3. Models.
    result = {"models_A": {}, "models_B": {}, "metadata": metadata}

    for scen_tag, key in [("scenario_A", "models_A"), ("scenario_B", "models_B")]:
        scen_dir = pkg / scen_tag
        if not scen_dir.exists():
            print(f"[LOAD] WARNING: {scen_tag} folder missing; skipped")
            continue

        for name in model_order:
            pre_path = scen_dir / "preprocess" / f"{name}_preprocess.joblib"
            xgb_path = scen_dir / "xgboost_model" / f"{name}_xgboost.joblib"
            if not pre_path.exists() or not xgb_path.exists():
                print(f"[LOAD] WARNING: {scen_tag}/{name} files missing; skipped")
                continue

            pre = joblib.load(pre_path)
            clf = joblib.load(xgb_path)
            result[key][name] = Pipeline(steps=[("preprocess", pre), ("clf", clf)])
            print(f"[LOAD] OK {scen_tag}/{name}")

        imp_path = scen_dir / "imputer.joblib"
        if imp_path.exists():
            imp     = joblib.load(imp_path)
            imp_key = "imputer_A" if scen_tag == "scenario_A" else "imputer_B"
            result[imp_key] = imp
            print(f"[LOAD] OK {scen_tag}/imputer  (include_cap={imp.include_cap})")

    return result


def predict(
    pkg: dict,
    df_new: pd.DataFrame,
    scenario: str = "B",
    model_name: str = "base_lsm_cap",
) -> np.ndarray:
    """Return positive-class prediction probabilities from a loaded package."""
    from two_tier_imputer import add_fibroscan_date_numeric

    imp      = pkg[f"imputer_{scenario}"]
    pipe     = pkg[f"models_{scenario}"][model_name]
    features = pkg["metadata"]["model_specs"][model_name]

    df_prep = df_new.copy()
    if scenario == "B" and "Fibroscan_Date_numeric" not in df_prep.columns:
        df_prep = add_fibroscan_date_numeric(df_prep)

    df_imp = imp.transform(df_prep)
    return pipe.predict_proba(df_imp[features])[:, 1]


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="CKD XGBoost model package loader")
    parser.add_argument("--pkg_dir", required=True, help="models/ directory path")
    parser.add_argument("--cpu", action="store_true", help="Ignored; kept for CLI compatibility")
    sys.argv = ['', '--pkg_dir', '/home/ubuntu/xgboost_CKD_output/models']
    args = parser.parse_args()

    pkg = load_package(args.pkg_dir, map_to_cpu=False)

    print("\n[SUMMARY] Load complete")
    for scen in ["A", "B"]:
        key = f"models_{scen}"
        if key in pkg and pkg[key]:
            print(f"  Scenario {scen}: {list(pkg[key].keys())}")


In [ ]:
import joblib
joblib.dump({
    'pred_test_A': pred_test_A, 'pred_test_B': pred_test_B,
    'pred_ext_A':  pred_ext_A,  'pred_ext_B':  pred_ext_B,
    'y_test_A':    y_test_A,    'y_test':      y_test,
    'y_ext_A':     y_ext_A,     'y_ext':       y_ext,
}, '/home/ubuntu/xgboost_CKD_output/predictions_cache.joblib')

In [ ]:
%whos

In [ ]:
print(pkg.keys())

In [ ]:
import inspect
print(inspect.getsource(predict))

In [ ]:
import sys
from sklearn.model_selection import train_test_split
sys.path.insert(0, '/home/ubuntu/xgboost_CKD_output/models/src')
from two_tier_imputer import is_cap_era, add_fibroscan_date_numeric, coerce_numeric

# ── 상수 ──────────────────────────────────────────────────────────────────
HOSP_INTERNAL = 10
HOSP_EXTERNAL = 20
RANDOM_STATE  = 42
MAX_XGBOOST_N  = None

# ── 1. 데이터 로드 ─────────────────────────────────────────────────────────
df_raw = pd.read_excel('/home/ubuntu/Fibroscan_CKD_2025.xlsx')

# ── 2. notebook 1과 동일한 전처리 ─────────────────────────────────────────
# df_final 생성 과정: Follow-up 있는 환자만, y 라벨 생성
df_final = df_raw[df_raw['Follow-up_duration'].notna()].copy()
df_final['y'] = ((df_final['CKD'] == 1) & (df_final['TTO_CKD'] <= 1825)).astype(int)

# ── 3. 분리 ───────────────────────────────────────────────────────────────
internal = df_final[df_final['Hospital'] == HOSP_INTERNAL].copy()
external = df_final[df_final['Hospital'] == HOSP_EXTERNAL].copy()

idx_train, idx_test = train_test_split(
    internal.index.values, test_size=0.2,
    random_state=RANDOM_STATE, stratify=internal['y'].values
)
df_train = internal.loc[idx_train].copy()
df_test  = internal.loc[idx_test].copy()
df_ext   = external.copy()

y_test = df_test['y'].astype(int).values
y_ext  = df_ext['y'].astype(int).values

# ── 4. Scenario A: CAP-era만 ──────────────────────────────────────────────
df_test_A = df_test[is_cap_era(df_test)].copy()
df_ext_A  = df_ext[is_cap_era(df_ext)].copy()
y_test_A  = df_test_A['y'].astype(int).values
y_ext_A   = df_ext_A['y'].astype(int).values

# ── 5. 예측 생성 ──────────────────────────────────────────────────────────
MODEL_ORDER = ['base', 'base_lsm', 'base_lsm_cap',
               'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']

pred_test_A = {m: predict(pkg, df_test_A, scenario='A', model_name=m) for m in MODEL_ORDER}
pred_ext_A  = {m: predict(pkg, df_ext_A,  scenario='A', model_name=m) for m in MODEL_ORDER}
pred_test_B = {m: predict(pkg, df_test,   scenario='B', model_name=m) for m in MODEL_ORDER}
pred_ext_B  = {m: predict(pkg, df_ext,    scenario='B', model_name=m) for m in MODEL_ORDER}

print("[OK] 예측 완료")
print(f"  test_A n={len(y_test_A)}, ext_A n={len(y_ext_A)}")
print(f"  test_B n={len(y_test)},   ext_B n={len(y_ext)}")